## Task 4.2: SCAFFOLD - Stochastic Controlled Averaging
**Goal: Use control variates to correct client drift**

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from scriptsfl.data_utils import load_dataset, create_dirichlet_split, get_dataloaders
from scriptsfl.models import get_model
from scriptsfl.federated_utils import get_model_weights, set_model_weights, evaluate_model
from scriptsfl.server import Server
from scriptsfl.results_utils import save_results, plot_comparison, print_summary

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print("="*80)
print("TASK 4.2: SCAFFOLD IMPLEMENTATION")
print("="*80)

#%% Configuration
CONFIG = {
    'dataset': 'cifar10',
    'num_clients': 5,
    'num_rounds': 50,
    'local_epochs': 5,
    'batch_size': 32,
    'lr': 0.01,
    'momentum': 0.0,  # SCAFFOLD typically uses momentum=0
    'alpha': 0.1,  # High heterogeneity
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

#%% Load non-IID data (same as Task 4.1)
train_dataset, test_dataset = load_dataset(CONFIG['dataset'])
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

client_indices = create_dirichlet_split(train_dataset, CONFIG['num_clients'], alpha=CONFIG['alpha'])
client_loaders = get_dataloaders(train_dataset, client_indices, batch_size=CONFIG['batch_size'])

## SCAFFOLD classes


In [ ]:
class SCAFFOLDClient:
    """
    SCAFFOLD Client with control variates

    Key idea: Maintain control variates c_i (local) and c (global)
    to correct gradient drift
    """

    def __init__(self, client_id, data_loader, model, device='cpu'):
        self.client_id = client_id
        self.data_loader = data_loader
        self.device = device
        self.model = copy.deepcopy(model).to(device)
        self.data_size = len(data_loader.dataset)

        # TODO: Initialize control variates
        # c_local: client-specific control variate (same shape as model params)
        # c_global: global control variate (will be received from server)

        # Initialize to zeros
        self.c_local = [torch.zeros_like(p) for p in self.model.parameters()]
        self.c_global = None  # Will be set by server

    def train_local(self, epochs, lr):
        """
        Train with SCAFFOLD gradient correction

        SCAFFOLD modifies gradients as:
        grad_corrected = grad + (c_global - c_local)
        """
        self.model.train()
        criterion = nn.CrossEntropyLoss()

        # TODO: Store initial weights for control variate update
        initial_weights = get_model_weights(self.model)

        # Training loop
        for epoch in range(epochs):
            for data, target in self.data_loader:
                data, target = data.to(self.device), target.to(self.device)

                # TODO: Forward pass
                # TODO: Compute loss
                # TODO: Backward pass

                # TODO: SCAFFOLD CORRECTION - modify gradients before step
                # For each parameter:
                #   grad = grad + (c_global - c_local)

                # TODO: Take optimizer step with corrected gradients

                pass  # Remove this when implementing

        # TODO: Update client control variate
        # Formula: c_local_new = c_local - (1/(K*lr)) * (theta_new - theta_global)
        # where K is number of local steps

        final_weights = get_model_weights(self.model)

        # TODO: Compute delta_c = c_local_new - c_local_old
        # This will be sent to server

        delta_c = None  # TODO: Compute this

        return {'client_id': self.client_id, 'delta_c': delta_c}

    def get_weights(self):
        return get_model_weights(self.model)

    def set_weights(self, weights):
        set_model_weights(self.model, weights)

    def set_global_control(self, c_global):
        self.c_global = c_global

    def get_data_size(self):
        return self.data_size

In [ ]:
#%% SCAFFOLD Server Implementation

class SCAFFOLDServer(Server):
    """
    Server for SCAFFOLD

    Maintains global control variate and updates it each round
    """

    def __init__(self, global_model, clients, test_loader, device='cpu'):
        super().__init__(global_model, clients, test_loader, device)

        # TODO: Initialize global control variate to zeros
        self.c_global = [torch.zeros_like(p) for p in global_model.parameters()]

    def broadcast_controls(self, clients):
        """
        Send global control variate to clients
        """
        for client in clients:
            # TODO: Send copy of c_global to each client
            client.set_global_control([c.clone() for c in self.c_global])

    def aggregate_controls(self, client_delta_c_list):
        """
        Update global control variate

        c_global = c_global + (1/M) * sum(delta_c_i)
        """
        # TODO: Average all client delta_c and update c_global
        pass

    def train_round(self, local_epochs, lr):
        """
        One round of SCAFFOLD training
        """
        # Select clients
        selected_clients = self.select_clients(fraction=1.0)

        # Broadcast model AND controls
        self.broadcast_weights(selected_clients)
        self.broadcast_controls(selected_clients)

        # Local training
        delta_c_list = []
        for client in selected_clients:
            stats = client.train_local(epochs=local_epochs, lr=lr)
            delta_c_list.append(stats['delta_c'])

        # Aggregate models (standard FedAvg aggregation)
        self.aggregate(selected_clients)

        # TODO: Aggregate control variates
        self.aggregate_controls(delta_c_list)

        # Evaluate
        test_acc, test_loss = self.evaluate()

        return {'test_accuracy': test_acc, 'test_loss': test_loss}

In [ ]:
#%% Run Experiments

print("\n--- Running SCAFFOLD ---")

# TODO: Initialize model and clients
model = get_model('simplecnn', num_classes=10, dataset=CONFIG['dataset'])
clients = [SCAFFOLDClient(i, client_loaders[i], model, CONFIG['device'])
           for i in range(CONFIG['num_clients'])]

# TODO: Create server and train
server = SCAFFOLDServer(model, clients, test_loader, CONFIG['device'])

# TODO: Implement training loop
# server.train(...) or manual loop

# TODO: Compare with FedAvg baseline from Task 4.1

In [ ]:
print("\n" + "="*80)
print("TASK 4.2 IMPLEMENTATION NOTES")
print("="*80)
print("SCAFFOLD is more complex - key steps:")
print("1. Maintain control variates c_i (per client) and c (global)")
print("2. Correct gradients during local training: grad += (c - c_i)")
print("3. Update c_i after local training based on weight change")
print("4. Server updates c as average of client controls")
print("5. SCAFFOLD requires sending c vectors (doubles communication)")
print("\nExpected result: SCAFFOLD should significantly outperform FedAvg")
print("under high heterogeneity, often matching IID performance")